# DistilRoBERTa financial sentiment

This notebook evaluates a smaller pretrained checkpoint first and optionally fine-tunes it.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.text_utils import load_data, metrics_table, set_seed, summarize_splits
from utils.transformer_utils import build_transformer_trainer, evaluate_transformer_checkpoint

set_seed(42)

I0000 00:00:1777752266.185031  261091 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777752266.214673  261091 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777752266.776013  261091 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777752266.776267  261091 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will 

In [2]:
MODEL_NAME = "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis"
MODEL_LABEL_ORDER = ["negative", "neutral", "positive"]
df = load_data()
display(summarize_splits(df))
MODEL_NAME

,rows,mean_words,negative,neutral,positive
split,,,,,
test,23566,17.30,5817,7306,10443
train,77589,18.01,18856,25403,33330
validation,23567,17.36,5722,7261,10584


'mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis'

In [3]:
results = evaluate_transformer_checkpoint(MODEL_NAME, MODEL_LABEL_ORDER, df, split="test", max_length=128, batch_size=32)
display(metrics_table(results))
display(results["report"])
display(results["confusion_matrix"])
results["predictions"].head()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

evaluating mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis:   0%|          | 0/737 [00:00<?,…

,metric,value
0,accuracy,0.623610
1,macro_precision,0.674149
2,macro_recall,0.631712
3,macro_f1,0.628225
4,weighted_f1,0.628402


,precision,recall,f1-score,support
negative,0.760074,0.567475,0.649803,5817.00000
neutral,0.478297,0.787298,0.595076,7306.00000
positive,0.784077,0.540362,0.639796,10443.00000
accuracy,0.623610,0.623610,0.623610,0.62361
macro avg,0.674149,0.631712,0.628225,23566.00000
weighted avg,0.683353,0.623610,0.628402,23566.00000


,negative,neutral,positive
negative,3301,1917,599
neutral,599,5752,955
positive,443,4357,5643


,text,label,prediction
0,hqge ltnc hbrm enzc eeenf halb azfl maxd mmex ...,positive,neutral
1,econx november nonfarm private payrolls k vs k...,positive,neutral
2,regulatory news the nomination committee of cy...,neutral,neutral
3,amazon labor union is now seeking to represent...,positive,neutral
4,greene king s third quarter sales boosted by f...,positive,positive


In [4]:
trainer, datasets, tokenizer = build_transformer_trainer(
    df,
    model_name=MODEL_NAME,
    model_label_order=MODEL_LABEL_ORDER,
    output_dir=str(ROOT / "artifacts" / "distilroberta_financial"),
    max_length=128,
    epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
)

trainer.train()
trainer.evaluate(datasets["test"])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Map:   0%|          | 0/77589 [00:00<?, ? examples/s]

Map:   0%|          | 0/23567 [00:00<?, ? examples/s]

Map:   0%|          | 0/23566 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.576946,0.365461,0.862774,0.857794,0.861869,0.859403,0.863285
2,0.333296,0.324526,0.890737,0.886762,0.886374,0.886400,0.890611


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0.333296,0.325962,2,0.890181,0.886776,0.886403,0.886321,0.890012


{'eval_loss': 0.3259618878364563,
 'eval_accuracy': 0.8901807689043537,
 'eval_macro_precision': 0.8867760262289858,
 'eval_macro_recall': 0.8864032684317974,
 'eval_macro_f1': 0.886320604011956,
 'eval_weighted_f1': 0.8900116048355103}